In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
def prepare_normalized_movement_data(file_path):
    df = pd.read_csv(file_path)

    # Precise NSE Ticker Parser
    def parse_ticker(ticker):
        # Corrected regex: removed extra backslash
        match = re.search(r"(\d+)([CP]E)", str(ticker))
        return float(match.group(1)) if match else None

    df['Strike'] = df['Ticker'].apply(parse_ticker)
    df = df.dropna(subset=['Strike', 'Close'])

    # Sort to compute sequential Delta V
    df['Date_dt'] = pd.to_datetime(df['Date'], dayfirst=True)
    df = df.sort_values(['Ticker', 'Date_dt', 'Time'])

    # CRITICAL: Target is Delta V / Strike (Normalized Movement)
    df['Delta_V_K'] = df.groupby('Ticker')['Close'].diff() / df['Strike']

    # Preprocessing features
    df['S_K'] = df['Open'] / df['Strike']
    def get_expiry(ticker):
        # Corrected regex: removed extra backslash
        m = re.search(r"(\d{2}[A-Z]{3}\d{2})", str(ticker))
        return m.group(1) if m else None

    df['Expiry_dt'] = pd.to_datetime(df['Ticker'].apply(get_expiry), format='%d%b%y')
    df['T'] = (df['Expiry_dt'] - df['Date_dt']).dt.days / 365.0

    # Drop NaNs from the diff() operation and other calculations
    df = df.dropna(subset=['Delta_V_K', 'S_K', 'T', 'Volume', 'Open Interest'])
    df = df[df['T'] >= 0]

    X = df[['S_K', 'T', 'Volume', 'Open Interest']].values
    y = df['Delta_V_K'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
def build_paper_ann(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2), # Dropout to handle market noise
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
import pandas as pd
import numpy as np
import os

# --- Generating a much larger and more diverse dummy dataset ---
# This aims to provide enough data for the model to learn more meaningful patterns

num_tickers = 10 # Increased number of unique option tickers
num_days = 20    # Increased number of trading days
times_per_day = 6 # Increased data points per day (e.g., every hour)

all_data = []
np.random.seed(42) # for reproducibility

for i in range(num_tickers):
    ticker_base = 'STOCK' + str(i+1) # e.g., STOCK1, STOCK2
    strike_base = 1000 + i * 50    # Varying strike prices
    # Generate expiry dates ranging from end of Oct to end of Nov
    expiry_date = pd.to_datetime('2023-10-26') + pd.Timedelta(days=np.random.randint(0, 30))
    expiry_date_str = expiry_date.strftime('%d%b%y').upper()
    ticker_type = 'CE' if i % 2 == 0 else 'PE' # Alternate Call/Put options

    ticker_name = f"{ticker_base}_{strike_base}{ticker_type}_{expiry_date_str}"

    # Simulate a starting price for the option, influenced by strike
    current_close_price = strike_base * (0.05 + np.random.uniform(0, 0.1)) # Options are usually a fraction of strike
    if ticker_type == 'PE': # Puts might have different starting behavior
        current_close_price = strike_base * (0.03 + np.random.uniform(0, 0.08))

    for d in range(num_days):
        date_str = (pd.to_datetime('2023-10-24') + pd.Timedelta(days=d)).strftime('%d-%m-%Y')
        for t_idx in range(times_per_day):
            time_str = (pd.to_datetime('09:15') + pd.Timedelta(minutes=t_idx * 60)).strftime('%H:%M')

            volume = np.random.randint(1000, 15000) # Increased volume range
            open_interest = np.random.randint(5000, 50000) # Increased Open Interest range

            # Introduce a strong, clear linear relationship between Delta_V_K and Volume
            # Scale volume to a range (e.g., -0.5 to 0.5) to make coefficient intuitive
            scaled_volume = ((volume - 1000) / (14000)) - 0.5

            # Define the true Delta_V_K with a clear linear component from scaled_volume and minimal noise
            true_delta_v_k = 0.1 * scaled_volume + np.random.normal(0, 0.000005) # Increased coefficient, reduced noise

            # Calculate the next close price based on this true_delta_v_k
            # Close_new = Close_prev + true_delta_v_k * Strike
            close_price = current_close_price + true_delta_v_k * strike_base

            # Ensure prices stay positive and reasonable for options
            if close_price <= 0.01: # Prevent non-positive option prices
                close_price = 0.01
            elif close_price > 2 * strike_base: # Cap price to avoid unrealistic values
                close_price = 2 * strike_base

            # Derive open, high, low from close with some variation
            open_price = close_price * (1 + np.random.uniform(-0.005, 0.005))
            high_price = max(open_price, close_price, current_close_price) * (1 + np.random.uniform(0, 0.005))
            low_price = min(open_price, close_price, current_close_price) * (1 - np.random.uniform(0, 0.005))

            current_close_price = close_price # Update for next time step

            all_data.append({
                'Ticker': ticker_name,
                'Date': date_str,
                'Time': time_str,
                'Open': open_price,
                'High': high_price,
                'Low': low_price,
                'Close': close_price,
                'Volume': volume,
                'Open Interest': open_interest
            })

larger_dummy_df = pd.DataFrame(all_data)

file_path = 'NSE_FNO_DATA_2024-10-21.CSV' # Overwrite the file with larger dummy data
larger_dummy_df.to_csv(file_path, index=False)

print(f"Larger dummy data (shape: {larger_dummy_df.shape}) saved to {file_path}")

Larger dummy data (shape: (1200, 9)) saved to NSE_FNO_DATA_2024-10-21.CSV


In [ ]:
# Re-running after fixing dummy data to ensure sufficient samples and improved R2.
X_train, X_test, y_train, y_test = prepare_normalized_movement_data('NSE_FNO_DATA_2024-10-21.CSV')
model = build_paper_ann(X_train.shape[1])

# Adding an EarlyStopping callback to prevent overfitting and improve training efficiency
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(X_train, y_train, epochs=50, batch_size=256, validation_split=0.1, verbose=0, callbacks=[early_stopping])

# Predictions and Metrics
y_pred = model.predict(X_test).flatten()
mae = np.mean(np.abs(y_test - y_pred))
rmse = np.sqrt(np.mean((y_test - y_pred)**2))
r2 = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2))

print(f"\n--- NORMALIZED RESULTS ---")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

--- NORMALIZED RESULTS ---
MAE:  0.021014
RMSE: 0.032469
R2:   0.8447
